# OTEL Spans — Silver Pipeline

Spark Declarative Pipeline that creates silver tables from raw OTEL span data.

| Table | Kind | Description |
| --- | --- | --- |
| `spans_flattened` | Streaming | All spans with timestamps, duration, K8s hierarchy, and service identity |
| `spans_service_calls` | Streaming | HTTP & gRPC server spans with route, status code, and latency |
| `spans_errors` | Streaming | Error spans for monitoring and alerting |
| `spans_db_operations` | Streaming | Database spans (PostgreSQL, Redis) with query text |
| `spans_messaging` | Streaming | Kafka producer/consumer spans with partition, offset, and queue lag |
| `spans_service_dependency_map` | Materialized View | Caller→callee edges with call counts and avg duration |

In [ ]:
import dlt
from pyspark.sql import functions as F

source_catalog = spark.conf.get("source_catalog")
source_schema = spark.conf.get("source_schema")
table_prefix = spark.conf.get("table_prefix")
source_table = f"{source_catalog}.{source_schema}.{table_prefix}_otel_spans"

In [ ]:
@dlt.table(
    comment="All OTel spans with flattened attributes, duration, and K8s resource hierarchy",
)
def spans_flattened():
    return (
        spark.readStream.table(source_table)
        .filter(F.col("start_time_unix_nano").isNotNull())
        .selectExpr(
            "trace_id",
            "span_id",
            "parent_span_id",
            "name AS span_name",
            "kind",
            "time AS start_timestamp",
            "TIMESTAMP_MICROS(CAST(end_time_unix_nano / 1000 AS BIGINT)) AS end_timestamp",
            "ROUND((end_time_unix_nano - start_time_unix_nano) / 1e6, 2) AS duration_ms",
            "status.code AS status_code",
            "status.message AS status_message",
            # K8s resource hierarchy
            "resource.attributes:['k8s.node.name']::string AS node",
            "resource.attributes:['k8s.namespace.name']::string AS namespace",
            "resource.attributes:['k8s.deployment.name']::string AS deployment",
            "resource.attributes:['k8s.pod.name']::string AS pod",
            "resource.attributes:['k8s.pod.uid']::string AS pod_uid",
            "resource.attributes:['k8s.container.name']::string AS container",
            "resource.attributes:['k8s.cluster.uid']::string AS cluster_uid",
            # Service identity
            "service_name AS service",
            "resource.attributes:['service.namespace']::string AS service_namespace",
            "resource.attributes:['service.instance.id']::string AS service_instance_id",
            "resource.attributes:['service.version']::string AS service_version",
            # Instrumentation
            "instrumentation_scope.name AS instrumentation_name",
            "instrumentation_scope.version AS instrumentation_version",
            # Common span attributes
            "attributes:['http.request.method']::string AS http_method",
            "COALESCE(attributes:['http.route']::string, attributes:['url.path']::string, attributes:['http.target']::string) AS http_route",
            "COALESCE(attributes:['http.response.status_code']::int, attributes:['http.status_code']::int) AS http_status_code",
            "attributes:['rpc.system']::string AS rpc_system",
            "attributes:['rpc.service']::string AS rpc_service",
            "attributes:['rpc.method']::string AS rpc_method",
            "attributes:['rpc.grpc.status_code']::int AS grpc_status_code",
            "attributes:['db.system']::string AS db_system",
            "attributes:['messaging.system']::string AS messaging_system",
            "attributes:['error.type']::string AS error_type",
            "attributes",
        )
    )

In [ ]:
@dlt.table(
    comment="HTTP and gRPC server spans — one row per inbound request with route, status, and latency",
)
def spans_service_calls():
    return (
        dlt.read_stream("spans_flattened")
        .filter(F.col("kind") == "SPAN_KIND_SERVER")
        .select(
            "trace_id",
            "span_id",
            "parent_span_id",
            "start_timestamp",
            "end_timestamp",
            "duration_ms",
            "status_code",
            "status_message",
            "service",
            "service_version",
            "node",
            "namespace",
            "deployment",
            "pod",
            # HTTP fields
            "span_name",
            "http_method",
            "http_route",
            "http_status_code",
            # gRPC fields
            "rpc_system",
            "rpc_service",
            "rpc_method",
            "grpc_status_code",
            # Protocol classification
            F.when(F.col("rpc_system").isNotNull(), F.lit("grpc"))
             .when(F.col("http_method").isNotNull(), F.lit("http"))
             .otherwise(F.lit("other"))
             .alias("protocol"),
            # Is this an error?
            F.when(
                (F.col("status_code") == "STATUS_CODE_ERROR")
                | (F.col("http_status_code") >= 500)
                | (F.col("grpc_status_code").isin(2, 13, 14)),  # UNKNOWN, INTERNAL, UNAVAILABLE
                F.lit(True),
            ).otherwise(F.lit(False)).alias("is_error"),
        )
    )

In [ ]:
@dlt.table(
    comment="Error spans for monitoring and alerting — any span with STATUS_CODE_ERROR",
)
def spans_errors():
    return (
        dlt.read_stream("spans_flattened")
        .filter(F.col("status_code") == "STATUS_CODE_ERROR")
        .select(
            "trace_id",
            "span_id",
            "parent_span_id",
            "span_name",
            "kind",
            "start_timestamp",
            "duration_ms",
            "status_message",
            "error_type",
            "service",
            "service_version",
            "node",
            "namespace",
            "deployment",
            "pod",
            "http_method",
            "http_route",
            "http_status_code",
            "rpc_service",
            "rpc_method",
            "grpc_status_code",
        )
    )

In [ ]:
@dlt.table(
    comment="Database operation spans (PostgreSQL, Redis) with query text and latency",
)
def spans_db_operations():
    return (
        dlt.read_stream("spans_flattened")
        .filter(F.col("db_system").isNotNull())
        .selectExpr(
            "trace_id",
            "span_id",
            "parent_span_id",
            "span_name",
            "start_timestamp",
            "duration_ms",
            "status_code",
            "status_message",
            "service",
            "node",
            "namespace",
            "pod",
            "db_system",
            "COALESCE(attributes:['db.namespace']::string, attributes:['db.name']::string) AS db_name",
            "COALESCE(attributes:['db.query.text']::string, attributes:['db.statement']::string) AS db_query",
            "attributes:['db.user']::string AS db_user",
            "attributes:['db.redis.database_index']::int AS redis_db_index",
            "attributes:['server.address']::string AS server_address",
            "attributes:['server.port']::int AS server_port",
        )
    )

In [ ]:
@dlt.table(
    comment="Kafka producer and consumer spans with partition, offset, and queue lag",
)
def spans_messaging():
    return (
        dlt.read_stream("spans_flattened")
        .filter(F.col("messaging_system").isNotNull())
        .selectExpr(
            "trace_id",
            "span_id",
            "parent_span_id",
            "span_name",
            "kind",
            "start_timestamp",
            "duration_ms",
            "status_code",
            "service",
            "node",
            "namespace",
            "pod",
            "messaging_system",
            "attributes:['messaging.operation']::string AS messaging_operation",
            "attributes:['messaging.destination.name']::string AS destination",
            "COALESCE(attributes:['messaging.destination.partition.id']::int, attributes:['messaging.kafka.destination.partition']::int) AS partition_id",
            "attributes:['messaging.kafka.message.offset']::bigint AS message_offset",
            "attributes:['messaging.kafka.consumer.group']::string AS consumer_group",
            "attributes:['messaging.client_id']::string AS client_id",
            "attributes:['messaging.message.body.size']::int AS message_body_size",
            "attributes:['messaging.batch.message_count']::int AS batch_message_count",
            "attributes:['kafka.record.queue_time_ms']::double AS queue_time_ms",
        )
    )

In [ ]:
@dlt.table(
    comment="Service dependency edges — caller to callee with call counts and avg duration",
)
def spans_service_dependency_map():
    # CLIENT spans represent outbound calls: the resource's service is the caller,
    # and the peer service (or rpc_service) is the callee.
    client_spans = (
        spark.read.table(source_table)
        .filter(
            (F.col("kind") == "SPAN_KIND_CLIENT")
            & F.col("start_time_unix_nano").isNotNull()
        )
        .selectExpr(
            "service_name AS caller_service",
            "COALESCE("
            "  attributes:['peer.service']::string,"
            "  attributes:['rpc.service']::string,"
            "  attributes:['db.system']::string,"
            "  attributes:['messaging.destination.name']::string,"
            "  attributes:['server.address']::string,"
            "  attributes:['net.peer.name']::string,"
            "  name"
            ") AS callee_service",
            "CASE"
            "  WHEN attributes:['rpc.system']::string IS NOT NULL THEN 'grpc'"
            "  WHEN attributes:['db.system']::string IS NOT NULL THEN 'database'"
            "  WHEN attributes:['messaging.system']::string IS NOT NULL THEN 'messaging'"
            "  WHEN COALESCE(attributes:['http.request.method']::string, attributes:['http.method']::string) IS NOT NULL THEN 'http'"
            "  ELSE 'other'"
            " END AS protocol",
            "ROUND((end_time_unix_nano - start_time_unix_nano) / 1e6, 2) AS duration_ms",
            "CASE WHEN status.code = 'STATUS_CODE_ERROR' THEN 1 ELSE 0 END AS is_error",
        )
        .filter(F.col("caller_service").isNotNull() & F.col("callee_service").isNotNull())
    )

    return (
        client_spans
        .groupBy("caller_service", "callee_service", "protocol")
        .agg(
            F.count("*").alias("call_count"),
            F.round(F.avg("duration_ms"), 2).alias("avg_duration_ms"),
            F.round(F.expr("percentile(duration_ms, 0.5)"), 2).alias("p50_duration_ms"),
            F.round(F.expr("percentile(duration_ms, 0.95)"), 2).alias("p95_duration_ms"),
            F.round(F.expr("percentile(duration_ms, 0.99)"), 2).alias("p99_duration_ms"),
            F.sum("is_error").alias("error_count"),
            F.round(F.avg("is_error") * 100, 2).alias("error_rate_pct"),
        )
    )